## Search Notebooks for Lab Results & Clinic Activity

In [ ]:
import sys
sys.path.append('..')
from credentials import *

from elasticsearch_utils import *

import duckdb
import pandas as pd
import numpy as np
import re

data_path = "../data/"
raw_data_path = data_path+'raw_data/'

## Load Patients of Interest

In [ ]:
cols = ['master_person_id', 'patient_identifier3', 'patient_identifier4', 'patient_identifier2']

inclusion_patients_df = pd.read_csv(os.path.join(data_path, "chronic_kidney_disease_refined_inclusion_patients.csv"))[cols]

## Lab Results

#### Search

### OMOP Lab Result Search

In [ ]:
with duckdb.connect() as conn:
    omop_lab_results_df = conn.execute("""ATTACH omop.duckdb as PROD; USE PROD;
                              SELECT DISTINCT em.master_person_id
                                     , em.measurement_date
                                     , em.measurement_source_value_name AS measurement
                                     , em.value_as_number AS measurement_value
                                     , em.unit_source_value AS measurement_unit
                              FROM ext_measurement AS em
                                  INNER JOIN inclusion_patients_df AS ip
                                      ON em.master_person_id = ip.master_person_id
                              WHERE em.value_as_number IS NOT NULL AND
                                    (UPPER(measurement_source_value_name) = 'EGFR BY CKD-EPI (2009)' OR UPPER(measurement_source_value_name) = 'ESTIMATED GFR' OR UPPER(measurement_source_value_name) = 'GFR CALCULATED' OR
                                     UPPER(measurement_source_value_name) = 'HB' OR UPPER(measurement_source_value_name) = 'HB.' OR UPPER(measurement_source_value_name) = 'HAEMOGLOBIN.' OR
                                     UPPER(measurement_source_value_name) = 'ALBUMIN LEVEL' OR
                                     UPPER(measurement_source_value_name) = 'C REACTIVE PROTEIN LEVEL' OR
                                     UPPER(measurement_source_value_name) = 'CREATININE LEVEL' OR UPPER(measurement_source_value_name) = 'CREATININE' OR
                                     UPPER(measurement_source_value_name) LIKE '%ALBUMIN%CREATININE%URINE%' OR UPPER(measurement_source_value_name) LIKE '%URINE%ALBUMIN%CREATININE%' OR
                                     UPPER(measurement_source_value_name) = 'CREATININE LEVEL URINE' OR UPPER(measurement_source_value_name) = 'CREATININE LEVEL URINE.' OR
                                     UPPER(measurement_source_value_name) = 'PROTEIN LEVEL URINE' OR UPPER(measurement_source_value_name) = 'URINE PROTEIN' OR
                                     UPPER(measurement_source_value_name) = 'ALBUMINE LEVEL URINE' OR
                                     (UPPER(measurement_source_value_name) LIKE '%PROT%' AND UPPER(measurement_source_value_name) LIKE '%CREATIN%') OR
                                     UPPER(measurement_source_value_name) LIKE 'HBA1C%' OR
                                     UPPER(measurement_source_value_name) = 'CHOLESTEROL (TOTAL) LEVEL' OR
                                     UPPER(measurement_source_value_name) = 'POCT GLUCOSE METER' OR UPPER(measurement_source_value_name) = 'BLOOD GLUCOSE RANDOM' OR --UPPER(measurement_source_value_name) = 'GLUCOSE LEVEL' OR
                                     UPPER(measurement_source_value_name) = 'NT-PROBNP' OR
                                     UPPER(measurement_source_value_name) LIKE 'TROPONIN I%')
                              ORDER BY em.master_person_id, em.measurement_date, em.measurement_source_value_name;""").df()

### EPIC Lab Results Search

In [ ]:
# --- Connect to Elasticsearch ---
es = connect_elasticsearch(hosts=hosts, username=username, password=password, api_key=True)

In [ ]:
# --- Explore Fields in a Given Index ---
index = 'lab_results'
columns = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3', 'document_Name', 'document_CreatedWhen', 'document_Fields']

In [ ]:
id_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier4']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            id_number.append(i)

In [ ]:
for idx, row in enumerate(inclusion_patients_df['patient_identifier3']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            try:
                id_number.append(int(i))
            except:
                pass

In [ ]:
for idx, row in enumerate(inclusion_patients_df['patient_identifier2']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            id_number.append(int(i))

In [ ]:
n = 65536

In [ ]:
epic_labs_df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(id_number, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "filter": [
                    {
                        "bool": {
                            "should": [
                                { "terms": { "patient_activity_document_identifiers": sub_list }}
                                ],
                            "minimum_should_match": 1
                            }
                        },
                    {
                        "nested": {
                            "path": "document_Fields",
                            "query": {
                                "bool": {
                                    "should": [
                                        {
                                            "terms": {
                                                "document_Fields.label": [
                                                    "eGFR",
                                                    "eGFR by CKD-EPI (2009)",
                                                    "eGFR by MDRD",
                                                    "EGFR BY CKD -15 MINUTES",
                                                    "Glucose",
                                                    "Creatinine",
                                                    "Serum Creatinine",
                                                    "Albumin",
                                                    "Haemoglobin",
                                                    "CRP",
                                                    "C Reactive Protein (CRP)",
                                                    "HbA1C",
                                                    "HbA1c",
                                                    "HbA1c (DCCT)",
                                                    "HbA1c (IFCC)",
                                                    "Cholesterol",
                                                    "Cholesterol (Total)",
                                                    "Cholesterol, Total",
                                                    "Cholesterol Body Fluid",
                                                    "Urine Protein/Creatinine Ratio",
                                                    "Urine Albumin/Creatinine Ratio ",
                                                    "Urine Creatinine ",
                                                    "Urine Albumin ",
                                                    "Urine Protein ",
                                                    "Urine Creatinine",
                                                    "Urine Albumin",
                                                    "Urine Protein",
                                                    "Troponin I",
                                                    "High Sensitivity Troponin I",
                                                    "NT-proBNP"
                                                    ]
                                                }
                                            }
                                        ],
                                    "minimum_should_match": 1
                                    }
                                },
                            "score_mode": "none"
                            }
                        }
                    ]
                }
            }
        }

    temp_df = es_docs_to_df(es, index=index, query=query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(epic_labs_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        epic_labs_df = pd.concat([epic_labs_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {epic_labs_df.shape[0]:,}\n")

    i+=1

In [ ]:
nhs_number_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[1].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower(): 
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            nhs_number_dict[nhs_num]=row[0]

epic_labs_df['masterPersonId_3'] = epic_labs_df['patient_identifier3'].map(nhs_number_dict)

In [ ]:
gstt_num_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[2].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            gstt_num_dict[nhs_num]=row[0]

epic_labs_df['masterPersonId_4'] = epic_labs_df['patient_identifier4'].map(gstt_num_dict)

In [ ]:
epic_mrn_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[3].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            epic_mrn_dict[nhs_num]=row[0]

epic_labs_df['masterPersonId_2'] = epic_labs_df['patient_identifier2'].map(epic_mrn_dict)

In [ ]:
epic_labs_df.insert(0, 'master_person_id', epic_labs_df['masterPersonId_3'].combine_first(epic_labs_df['masterPersonId_4']).combine_first(epic_labs_df['masterPersonId_2']))

In [ ]:
cols = ['master_person_id', 'resultDate', 'label', 'valueNum', 'unitOfMeasure']

rename_dict = {'resultDate' : 'measurement_date', 'label' : 'measurement', 'valueNum' : 'measurement_value', 'unitOfMeasure' : 'measurement_unit'}

epic_labs_df_exploded = epic_labs_df.explode('document_Fields').reset_index(drop=True)

epic_lab_results_df = pd.concat(
    [
        epic_labs_df_exploded.drop(columns=['document_Fields']),
        epic_labs_df_exploded['document_Fields'].apply(pd.Series)
    ],
    axis=1
)

rel_tests = ["eGFR", "eGFR by CKD-EPI (2009)", "eGFR by MDRD", "EGFR BY CKD -15 MINUTES", "Glucose", "Serum Creatinine", "Hb",
             "Creatinine", "Albumin", "Haemoglobin", "CRP", "C Reactive Protein (CRP)", "HbA1C", "HbA1c", "HbA1c (DCCT)",
             "Cholesterol Body Fluid", "HbA1c (IFCC)", "Cholesterol", "Cholesterol (Total)", "Cholesterol, Total", "NT-proBNP",
             "Troponin I", "Urine Protein/Creatinine Ratio", "Urine Albumin/Creatinine Ratio ", "Urine Albumin/Creatinine Ratio",
             "High Sensitivity Troponin I", "Urine Creatinine", "Urine Albumin", "Urine Protein", "Urine Creatinine ", "Urine Albumin ", "Urine Protein "]

epic_lab_results_df = epic_lab_results_df[(epic_lab_results_df['label'].isin(rel_tests))&(epic_lab_results_df['valueNum'].notna())][cols].rename(columns=rename_dict).reset_index(drop=True)

del epic_labs_df_exploded, epic_labs_df, nhs_number_dict, gstt_num_dict, epic_mrn_dict, id_number, full_list, rel_tests, rename_dict, cols

### Combine & Clean Lab Results

In [ ]:
lab_results_df = pd.concat([omop_lab_results_df, epic_lab_results_df])

print(f'Number of Legacy/OMOP Results: {omop_lab_results_df.shape[0]:,}')
print(f'Number of EPIC Results: {epic_lab_results_df.shape[0]:,}')

del omop_lab_results_df, epic_lab_results_df

lab_results_df['measurement_date'] = pd.to_datetime(lab_results_df['measurement_date']).dt.date

lab_results_df = lab_results_df.sort_values(by=['master_person_id', 'measurement_date']).reset_index(drop=True)

print(f'Total Number of Results: {lab_results_df.shape[0]:,}')

lab_results_df.head()

In [ ]:
## Standardise Test Names
def lab_result_cleaning(x):
    measure = ''
    
    if x in ['EGFR BY CKD-EPI (2009)', 'Estimated GFR', 'eGFR by CKD-EPI (2009)', 'eGFR', 'GFR Calculated', 'eGFR by MDRD', 'GFR calculated']:
        measure = 'eGFR'
    elif x in ['Hb', 'HB', 'HB.', 'Haemoglobin.', 'Haemoglobin']:
        measure = 'Haemoglobin'
    elif x in ['Troponin I', 'Troponin I (poct)', 'High Sensitivity Troponin I']:
        measure = 'Troponin I'
    elif x in ['Albumin Level', 'Albumin']:
        measure = 'Albumin'
    elif x in ['POCT Glucose Meter', 'Blood Glucose Random', 'Glucose Level', 'Glucose']:
        measure = 'Random Glucose'
    elif x in ['Albumin Creatinine Ratio Level Urine', 'Urine albumin creatinine', 'Urine Albumin Creatinine', 'Albumin:Creatinine Ratio Level Urine',
               'Albumin;Creatinine Urine', 'Unprocessed Albumin Creatinine Urine', 'Urine Albumin/Creatinine Ratio ']:
        measure = 'Urine ACR'
    elif x in ['NT-proBNP', 'NT-PROBNP']:
        measure = 'NT-proBNP'
    elif x in ['HbA1c Level', 'HbA1c Level.', 'HbA1c New Units', 'HbA1C', 'HbA1c', 'HbA1c (IFCC)', 'HBA1C (ROCHE)', 'HBA1C (IFCC) (CALCULATED)', 'HbA1c (DCCT)']:
        measure = 'HbA1c'
    elif x in ['C Reactive Protein Level', 'C Reactive Protein (CRP)']:
        measure = 'CRP'
    elif x in ['Cholesterol (total) Level', 'Cholesterol (Total)', 'Cholesterol', 'Cholesterol Body Fluid']:
        measure = 'Cholesterol'
    elif x in ['Urine Protein Creatinine', 'Urine protein creatinine', 'Protein Creatinine Ratio Calculated', 'Urine Protein/Creatinine Ratio', 'Urine Albumin/Creatinine Ratio ']:
        measure = 'Urine PCR'
    elif x in ['Creatinine Level', 'Creatinine', 'CREATININE', 'Serum Creatinine']:
        measure = 'Creatinine'
    elif x in ['Creatinine Level Urine', 'Urine Creatinine ', 'Creatinine Level Urine.', 'Urine Creatinine']:
        measure = 'Urine Creatinine'
    elif x in ['Urine Albumin ']:
        return 'Urine Albumin'
    elif x in ['Protein Level Urine', 'URINE PROTEIN', 'Urine Protein']:
        return 'Urine Protein'

    return measure

lab_results_df['measurement_cleaned'] = lab_results_df['measurement'].apply(lambda x: lab_result_cleaning(x))

In [ ]:
## Standardise Unit Names
def unit_cleaning(x):
    units = x
    
    if x == 'umol/l':
        units = 'umol/L'
    elif x == 'mg/mmo':
        units = 'mg/mmol'
    elif x in ['mL/min/1.73m2', 'ml/min/1.73 m2']:
        units = 'mL/min/1.73m2'
    elif x in ['mL/min', 'ml/min']:
        units = 'mL/min'

    return units

lab_results_df['measurement_unit_cleaned'] = lab_results_df['measurement_unit'].apply(lambda x: unit_cleaning(x))

In [ ]:
lab_results_df.head()

### Export Data

In [ ]:
# --- Save Results ---
file_name = "20251118_lab_search_results.csv"

lab_results_df.to_csv(f"{raw_data_path}/{file_name}", index=False)
print("✅ Results saved.")

## Sandbox